<a href="https://colab.research.google.com/github/Xyrfo/Pembelajaran-Mesin/blob/main/JS09/tugas_lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Lab 1 - JS09 Approximate Nearest Neighbors

Lakukan percobaan pada **metric distance yang berbeda**, **1000 vs 1.000.000 data**, **2D vs 5D data**, untuk algoritma **ANNOY**, **FAISS**, dan **HNSW**. Catat performansinya (build time & query time) dalam bentuk tabel.

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 4.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 20.3 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=c9bffcf1949b408fdaf389168dd456dacba2398ddbfa8bd773a28a0e30784844
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=8c8e8275905c131eb778c470283ca528e08d993be64a8cef8d0314019b7c32bf
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [2]:
import numpy as np
import time
import pandas as pd
import faiss
from annoy import AnnoyIndex
import hnswlib

## Helper Benchmark

Setiap fungsi mengembalikan `(build_time, query_time)` dalam detik. Metrik yang dipakai: **Euclidean** dan **Cosine**.

In [3]:
def make_data(n, d, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.random((n, d)).astype('float32')
    q = rng.random((1, d)).astype('float32')
    return X, q


def bench_annoy(X, q, metric, k=10, n_trees=10):
    n, d = X.shape
    space = 'angular' if metric == 'cosine' else 'euclidean'
    index = AnnoyIndex(d, space)

    start = time.time()
    for i in range(n):
        index.add_item(i, X[i])
    index.build(n_trees)
    build_time = time.time() - start

    start = time.time()
    index.get_nns_by_vector(q[0], k)
    query_time = time.time() - start
    return build_time, query_time


def bench_faiss(X, q, metric, k=10):
    X_use = X.copy()
    q_use = q.copy()
    d = X.shape[1]

    if metric == 'cosine':
        faiss.normalize_L2(X_use)
        faiss.normalize_L2(q_use)
        index = faiss.IndexFlatIP(d)
    else:
        index = faiss.IndexFlatL2(d)

    start = time.time()
    index.add(X_use)
    build_time = time.time() - start

    start = time.time()
    index.search(q_use, k)
    query_time = time.time() - start
    return build_time, query_time


def bench_hnsw(X, q, metric, k=10):
    n, d = X.shape
    space = 'cosine' if metric == 'cosine' else 'l2'
    index = hnswlib.Index(space=space, dim=d)

    start = time.time()
    index.init_index(max_elements=n, ef_construction=100, M=16)
    index.add_items(X)
    index.set_ef(50)
    build_time = time.time() - start

    start = time.time()
    index.knn_query(q, k=k)
    query_time = time.time() - start
    return build_time, query_time

## Jalankan Percobaan

Kombinasi metrik (Euclidean, Cosine) x ukuran data/dimensi (1000/2D, 1000000/5D).

In [4]:
configs = [(1000, 2), (1000000, 5)]
metrics = ['euclidean', 'cosine']

rows = []
for n, d in configs:
    X, q = make_data(n, d)
    for metric in metrics:
        ab, aq = bench_annoy(X, q, metric)
        fb, fq = bench_faiss(X, q, metric)
        hb, hq = bench_hnsw(X, q, metric)

        rows.append({
            'Metrik': metric,
            'Jumlah Data/Dimensi': f'{n}/{d}D',
            'ANNOY (build/query)': f'{ab:.4f} / {aq:.6f}',
            'FAISS (build/query)': f'{fb:.4f} / {fq:.6f}',
            'HNSW (build/query)': f'{hb:.4f} / {hq:.6f}',
        })

df_result = pd.DataFrame(rows)
df_result

,Metrik,Jumlah Data/Dimensi,ANNOY (build/query),FAISS (build/query),HNSW (build/query)
0,euclidean,1000/2D,0.0227 / 0.000100,0.0001 / 0.000079,0.0271 / 0.000103
1,cosine,1000/2D,0.0301 / 0.000060,0.0000 / 0.000041,0.0210 / 0.000082
2,euclidean,1000000/5D,25.5627 / 0.000108,0.0135 / 0.009262,85.5142 / 0.000161
3,cosine,1000000/5D,27.0840 / 0.000125,0.0182 / 0.006182,79.0478 / 0.000124
